# MongoDB Connection Test

This notebook tests the connection to the MongoDB instance using the application settings.

In [13]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1. Allow imports from the parent 'app' directory
current_dir = Path.cwd()
parent_dir = current_dir.parent

if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

print(f"Working configuration context: {parent_dir}")

# 2. Load Environment Variables explicitly from .env.local or .env
# We prioritize .env.local if it exists
env_path = parent_dir / ".env.local"
if not env_path.exists():
    env_path = parent_dir / ".env"

if env_path.exists():
    load_dotenv(env_path)
    print(f"Loaded environment from: {env_path.name}")
else:
    print("Warning: No .env or .env.local file found!")

Working configuration context: c:\Users\Lenovo\GIT\cityHunter\vibe\hunterBack
Loaded environment from: .env.local


In [14]:
from motor.motor_asyncio import AsyncIOMotorClient
from app.core.config import settings

print(f"Target Database: {settings.DB_NAME}")


async def test_mongo_connection():
    client = None
    try:
        # Create a new client
        client = AsyncIOMotorClient(settings.MONGO_URI)

        # The 'ping' command is a lightweight way to check connectivity
        await client.admin.command("ping")
        print("✅ Successfully connected to MongoDB!")

        # List collections to verify database access
        db = client[settings.DB_NAME]
        collections = await db.list_collection_names()
        print(f"📂 Collections in '{settings.DB_NAME}': {collections}")
        return client, db

    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return None, None


# Run the async test and keep client/db for next cells
client, db = await test_mongo_connection()

Target Database: hunter_db
✅ Successfully connected to MongoDB!
📂 Collections in 'hunter_db': ['UserProfile']


### Query Documents
Now we can try to find documents in the collections.

In [15]:
if db is not None:
    # Get collections list again to be sure
    collections = await db.list_collection_names()

    for col_name in collections:
        print(f"--- Content of collection: {col_name} ---")
        # Get the collection object
        collection = db[col_name]

        # Find documents (limit to 5)
        cursor = collection.find({}).limit(5)

        # Convert cursor to list
        docs = await cursor.to_list(length=5)

        for doc in docs:
            print(doc)
else:
    print("Database connection not established.")

--- Content of collection: UserProfile ---


In [ ]:
# Clean up connection when done
if client:
    client.close()
    print("Closed MongoDB connection")